---
date: 2026-09-24
---

# How this site is built

This post has two jobs. It explains the design of the site, so someone else can run the same thing on their own domain. And it states the rules that keep the template generic, so a fork is a copy and not a find-and-replace. The [README](https://github.com/cantux/blog_template/blob/main/README.md) has the step-by-step; this is the why, and the list of what may change.

Recreating the site takes five things:

1. A copy of the template repo: one MyST config, three pages, and posts as Jupyter notebooks saved with their outputs.
2. One GitHub Actions workflow that builds the site with Node, builds the editable JupyterLite app with Python, and deploys to GitHub Pages. It holds no site-specific value.
3. Two settings on GitHub: Pages source set to GitHub Actions, and the domain.
4. A domain with four A records at the apex pointing at GitHub Pages. GitHub issues the certificate.
5. Optional: a private JupyterHub for posts that need a GPU or compiled packages, and a second repo for the posts so the hub can push without a key to the site. Nothing on the site points at the hub.

This site is built from the template at [github.com/cantux/blog_template](https://github.com/cantux/blog_template). File names below link to it on first mention.

## The design

Two systems, kept apart on purpose: a static site the public reads, and a private JupyterHub where the notebooks get written. Almost every decision below falls out of that split.

```{figure} images/site-design.svg
:alt: Author, hub on EC2, posts repo, site repo, GitHub Pages and visitor, with the four numbered steps of a publish between them.

The components, and the four steps from a saved notebook to a published page.
```

The numbered path is the whole authoring loop. ① The author writes a notebook in JupyterLab on the hub and saves it with its outputs. ② A commit and push from the hub lands in the posts repo; the hub holds a deploy key that opens that one repo and nothing else. ③ The posts repo's workflow starts the site repo's workflow with a token that can only run Actions; that workflow checks out the posts repo at its head, builds the site, and builds the JupyterLite copy. ④ GitHub Pages serves the result at the domain. Two halves, one arrow between them, and the arrow is a git push. Nothing on the published site points at the hub: no URL, no token, no CORS rule. That deletes the three problems a public notebook server would hand you: cross-origin access, anonymous kernel authentication, and hardening a machine that runs strangers' code.

A notebook is a document with a kernel attached, so publishing one means choosing where the code runs. The author's machine runs anything, but the reader gets a picture of a result. A server the author operates gives every reader a real kernel, and is the three problems above. The reader's browser runs Python compiled to WebAssembly for free, but only the packages Pyodide ships. This site uses the first and the third: every page shows the outputs the author committed, and a post can link to a copy of itself in JupyterLite, a full notebook interface that runs entirely in the reader's browser. Nothing on a page executes on its own.

So every post has the same shape: outputs committed, shown as saved. A post whose imports all exist in Pyodide can add the JupyterLite link. One that needs a GPU or a compiled package cannot, and shows its outputs only.

## The site is one config file

The generator is MyST (Jupyter Book 2). It renders `.ipynb` files with their saved outputs. No conversion step to write, no kernel to wire up.

[`myst.yml`](https://github.com/cantux/blog_template/blob/main/myst.yml) is the whole application — table of contents and theme:

```yaml
version: 1
project:
  title: Your Name
  toc:
    - file: index.md
    - file: about.ipynb
    - file: blog.ipynb
      children:
        - file: bootstrap.ipynb
site:
  template: book-theme
  options:
    logo_text: Your Name
    style: custom.css
```

- `toc` is the page list and the shared nav at once. There is no second menu to keep in sync, and a file that is not in the `toc` is not built.
- `logo_text` makes the header link the site name. `style` loads [`custom.css`](https://github.com/cantux/blog_template/blob/main/custom.css), the only stylesheet in the repo: three lines that hide the duplicate home entry in the sidebar.
- LaTeX math renders on every page (KaTeX): `$...$` in any markdown cell, nothing to install.

There is no application code. [`index.md`](https://github.com/cantux/blog_template/blob/main/index.md) is the landing page; [`about.ipynb`](https://github.com/cantux/blog_template/blob/main/about.ipynb) and [`blog.ipynb`](https://github.com/cantux/blog_template/blob/main/blog.ipynb) are notebooks, same as the dated posts under `posts/`. All of them are committed with their outputs, and that habit is what the build rests on.

## Committed outputs are the build contract

A post is saved with its outputs inside the `.ipynb`. Because the outputs are already in the file, the build never starts a kernel. It never installs a scientific Python stack, never resolves a dependency, never picks a version. It reads JSON and writes HTML. The core build is Node only — install `mystmd`, run `myst build --html` — and it cannot break tomorrow because a package you have never heard of published a bad release.

The trade is that the repo carries the results. An interactive Plotly figure is a few hundred kilobytes of JSON in the notebook, and the repo grows with every post. The sharper cost is that an output can go stale: edit the code, forget to re-run, and the page shows a number the code no longer produces. No build step catches that. Re-run before you push.

## Build and deploy

Every push to `main` runs one GitHub Actions workflow, [`.github/workflows/deploy.yml`](https://github.com/cantux/blog_template/blob/main/.github/workflows/deploy.yml), and it is the only thing that ever publishes. It starts by taking every submodule at its head:

```yaml
- uses: actions/checkout@v4
  with: {submodules: true}
- run: git submodule update --init --remote
```

So when `posts/` is a submodule, the site builds the posts repo's `main`, not the commit the site repo last recorded; the recorded pointer is bookkeeping. A push to the posts repo is then enough to publish, once that repo has a workflow of its own whose only step starts this one:

```yaml
- run: gh api -X POST repos/<user>/<site-repo>/actions/workflows/deploy.yml/dispatches -f ref=main
  env:
    GH_TOKEN: ${{ secrets.SITE_ACTIONS_TOKEN }}
```

The token is fine-grained: one repo, one permission, Actions read and write. It can start a build and nothing else. Call the REST endpoint, not `gh workflow run`; that command looks up the default branch over GraphQL, which refuses fine-grained tokens.

Then the build:

```yaml
- uses: actions/setup-node@v4
  with: {node-version: 24.x}
- run: |
    if [ -f CNAME ]; then
      echo "BASE_URL=" >> "$GITHUB_ENV"
    else
      echo "BASE_URL=/${GITHUB_REPOSITORY#*/}" >> "$GITHUB_ENV"
    fi
- run: npm install -g mystmd@1.10.1
- run: myst build --html
- run: cp robots.txt _build/html/robots.txt
```

There is no domain, no name, no site-specific value in it. That is what makes a fork a copy: the workflow reads the site's identity from files it can see.

`CNAME` holds the domain, one line, and is the only place the domain is written down. GitHub Pages itself does not read it; a workflow deploy takes the domain from the repo settings. The workflow reads it to settle `BASE_URL`: empty when the site lives at the root of a domain, `/reponame` when it lives at `user.github.io/reponame`. Get that wrong and every link and asset points at a path that is not there. The one fact that separates the two cases is whether a `CNAME` exists, so leave the file out and a fork publishes at a `github.io` path with nothing else changed.

[`robots.txt`](https://github.com/cantux/blog_template/blob/main/robots.txt) is a file rather than a line of shell because it is a decision someone will reverse. It ships as `Disallow: /`, which keeps a site under construction out of the search index. Delete that line when you want readers.

Python enters the same job once, in later steps, to build the editable JupyterLite app into `/lite/`:

```yaml
- run: pip install jupyterlite-core==0.8.4 jupyterlite-pyodide-kernel==0.8.6 jupyter-server==2.21.1
- run: python scripts/jlite_contents.py
- run: jupyter lite build --contents .jlite --output-dir _build/html/lite
```

The three are pinned on purpose. A JupyterLite build ships pre-built WebAssembly straight to the reader, so an unpinned bump quietly changes what their browser downloads and runs. [`scripts/jlite_contents.py`](https://github.com/cantux/blog_template/blob/main/scripts/jlite_contents.py) copies `posts/` into `.jlite/` and drops any cell containing a `/lite/` link, so the editable copy does not carry a link back to itself. A post points at its editable version with a bold link:

    **[In-browser notebook →](/lite/notebooks/index.html?path=my-post.ipynb)**

The last two steps upload `_build/html` and hand it to `actions/deploy-pages`. In the repo's Pages settings the source must be "GitHub Actions", not a branch; the custom domain is set in the same place.

## Domain, DNS, and TLS

Two records in one zone:

| Name | Type | Value |
|---|---|---|
| `yourdomain.com` | A | `185.199.108.153`, `.109.153`, `.110.153`, `.111.153` (GitHub Pages) |
| `hub` | A | the EC2 Elastic IP |

The apex has to be A records: a CNAME cannot coexist with the SOA and NS records every zone apex carries, which is why the four GitHub Pages addresses are listed instead of a pointer at `user.github.io`. GitHub provisions the certificate once the records resolve; tick "Enforce HTTPS" in the Pages settings when it goes green. The hub gets its own Let's Encrypt certificate, issued from its own server, and its record only exists if you run a hub.

Two names, two certificates, two issuers. The DNS zone is the only thing the site and the hub share.

## What changes in a fork, and what must not

Four kinds of thing change, and it helps to know which is which before editing.

**Who you are.** Title and header text in `myst.yml`, the landing page in `index.md`, the About page in `about.ipynb`. Pure content.

**Where it lives.** The `CNAME` file. No domain: leave it out, and the workflow handles the rest.

**What is on it.** Posts in `posts/`, as files or as a submodule of their own repo, and their lines in the listing in `blog.ipynb`. Keep this post if you like; a fork that carries the recipe lets the next person take it from you.

**The two settings that are not in the repo.** DNS records at your registrar, and Settings → Pages → Source: GitHub Actions. The second is the usual first-deploy failure: left on "Deploy from a branch", the workflow runs green and nothing appears.

[`setup.sh`](https://github.com/cantux/blog_template/blob/main/setup.sh) does the first two. Set three variables at the top, run it once, and it writes `CNAME` and puts your name into `myst.yml`, `index.md` and `about.ipynb`. The README has the exact sequence, including the two settings it cannot touch.

Everything else stays generic, and five rules keep it so:

1. The workflow holds no site-specific value. The domain comes from `CNAME`, the title from `myst.yml`, and every version that reaches a reader's browser is pinned.
2. Identity lives in five files: `CNAME`, `myst.yml`, `index.md`, `about.ipynb`, `blog.ipynb`, plus `.gitmodules` when the posts are a submodule. A fork changes nothing else.
3. Every notebook is committed with its outputs. The build never runs a kernel.
4. Nothing published points at the hub. No URL, no token, no CORS rule.
5. Cloud-specific hub values live only in the cloud overlay. The base values run in a local VM.

Because of rule 2, a fork can take later fixes from the template: add it as a git remote and pull, and only the files that carry your identity ever conflict. A change that breaks one of these rules is a change to the design, not to the content, and it goes in this post first.

## Optional: the authoring hub

Everything above is the site, and the site is complete. A laptop with Jupyter installed publishes to it.

The hub exists for the posts Pyodide cannot run. It is a private JupyterHub with real hardware behind it — a GPU when a post needs one, compiled packages, kernels that stay alive between cells. Nothing published points at it.

Two flavors live in the template, and both boot in a local Lima VM before anything touches AWS:

- [`tljh/`](https://github.com/cantux/blog_template/tree/main/tljh) — The Littlest JupyterHub on one EC2 instance. The install and HTTPS, in full:
  ```bash
  curl -L https://tljh.jupyter.org/bootstrap.py | sudo python3 - --admin <user>
  sudo tljh-config set https.enabled true
  sudo tljh-config set https.letsencrypt.email <your-email>
  sudo tljh-config add-item https.letsencrypt.domains hub.<yourdomain>
  sudo tljh-config reload proxy
  ```
- [`z2jh/`](https://github.com/cantux/blog_template/tree/main/z2jh) — Zero to JupyterHub on Kubernetes: a base values file plus overlays, `values-local.yaml` and `values-eks.yaml`. Nothing cloud-specific is allowed in the base file, which is what makes the local hub the same hub.

Two EC2 choices carry the design. The instance is x86_64, so it can be stopped, switched to `g4dn.xlarge` for a GPU session, and switched back — same disk, same IP; ARM types cannot make that swap. And it has an Elastic IP, so the `hub` record survives every stop and resize. The rest of the checklist — disk, metadata service, no IAM role, GitHub OAuth — is in the [runbook](https://github.com/cantux/blog_template/blob/main/tljh/README.md).

The writing loop: keep the posts in a repo of their own, mounted as `posts/`, and clone it on the hub with a deploy key scoped to that repo, write access only — never a personal SSH key on a cloud box. jupyterlab-git adds a Git tab to JupyterLab, so a push is a commit message and one click. The posts repo's workflow starts the site deploy, and the post is live a few minutes later. The hub never holds a key to the site repo, and the site repo holds no secret at all. The same runbook has the commands.

## Cost

| Item | Monthly |
|---|---|
| Site (GitHub Pages) | $0 |
| t3.small, always on | ~$15 |
| 30 GB gp3 + public IPv4 | ~$6 |
| GPU authoring (g4dn.xlarge) | ~$0.53/hr as used |
| Domain | ~$1 amortized |

A fork that stops before the hub pays only the first row.

## Conclusion

Two systems, kept apart. The reader's half is a static site: free, no server, nothing to harden. The author's half is a private hub: optional, about $21 a month, and unreachable from the site. The split is what makes each half cheap, and it is the same split that removed cross-origin access, anonymous kernel authentication and hostile code from the problem.

The template stays generic as long as the five rules hold: a generic workflow, identity in five files, outputs committed, nothing pointing at the hub, cloud values only in the overlay. A fork sets three variables and runs; the README has the sequence.